# 第13章：内存 Swizzle -- L2 缓存优化

## 前置知识
- 第09章：分块矩阵乘法基础
- 第12章：Block Pointer 与 Shared Memory
- GPU 缓存层级的基本概念

## 学习目标
- 理解朴素 block 调度顺序导致的 **L2 Cache 抖动 (thrashing)** 问题
- 掌握 **Swizzled** block 调度的原理和实现
- 理解 GROUP_SIZE_M 参数的意义
- 了解 `make_block_ptr` 的 `order` 参数对 Shared Memory 布局的影响
- 实现 Swizzled GEMM 并对比性能

## 对应 CUDA 代码
- `src/simt/03simt_smemT.cu` — Shared Memory 转置存储优化
- Swizzle 概念在 CUDA 中对应 block 调度优化和 smem 布局优化

In [1]:
import torch
import triton
import triton.language as tl

## 13.1 问题：L2 Cache 抖动

### 朴素的 Block 调度顺序

在标准的 2D grid 中，program (block) 按照 `(pid_m, pid_n)` 的线性顺序执行：

```
朴素调度: pid = pid_m * grid_n + pid_n 的顺序

矩阵 C 的 block 分布 (8x8 grid):
      pid_n:  0    1    2    3    4    5    6    7
pid_m:   ┌────┬────┬────┬────┬────┬────┬────┬────┐
  0      │  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │  ← 第一批 (0-7)
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  1      │  8 │  9 │ 10 │ 11 │ 12 │ 13 │ 14 │ 15 │  ← 第二批 (8-15)
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  2      │ 16 │ 17 │ 18 │ 19 │ 20 │ 21 │ 22 │ 23 │  ← 第三批
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  3      │ 24 │ 25 │ 26 │ 27 │ 28 │ 29 │ 30 │ 31 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  4      │ 32 │ 33 │ 34 │ 35 │ 36 │ 37 │ 38 │ 39 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  5      │ 40 │ 41 │ 42 │ 43 │ 44 │ 45 │ 46 │ 47 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  6      │ 48 │ 49 │ 50 │ 51 │ 52 │ 53 │ 54 │ 55 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  7      │ 56 │ 57 │ 58 │ 59 │ 60 │ 61 │ 62 │ 63 │
         └────┴────┴────┴────┴────┴────┴────┴────┘

数字 = 执行顺序 (pid)
```

### L2 Cache 抖动分析

```
同时运行 pid 0-7 (假设 GPU 有 8 个 SM):

         pid_n:  0    1    2    3    4    5    6    7
pid_m=0:      │  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │

这 8 个 program 需要的数据:
  A: 都需要 A 的 第0行块 (同一行) → 可以共享 L2 cache ✓
  B: 需要 B 的 第0,1,2,3,4,5,6,7列块 → 需要 B 的整行! 
     占用 L2 cache: N * BLOCK_K * sizeof(half) = 大量!

下一轮, pid 8-15 (pid_m=1):
  A: 需要 A 的 第1行块 → 第0行块的 cache 被驱逐!
  B: 又需要 B 的 第0,1,2,...,7列块 → 之前的 B 数据可能还在 cache 中
     但 A 的数据被换出了!

问题: 每次切换 pid_m, A 的 cache 被完全替换
     如果 B 的整行放不进 L2, B 的数据也会被反复换入换出
     → L2 Cache Thrashing!
```

### 数据访问模式分析

```
朴素调度的数据访问:

矩阵 A (M x K):           矩阵 B (K x N):
┌───────────────┐          ┌──┬──┬──┬──┬──┬──┬──┬──┐
│ ████████████  │ pid 0-7  │██│██│██│██│██│██│██│██│ pid 0-7
│               │ 都读这行  │  │  │  │  │  │  │  │  │ 每个读一列
│               │          │  │  │  │  │  │  │  │  │
│               │          │  │  │  │  │  │  │  │  │
└───────────────┘          └──┴──┴──┴──┴──┴──┴──┴──┘
                              ↑ 需要 B 的全部 N 列在 L2 中!
                              (如果放不下就 thrashing)
```

## 13.2 Swizzled Block 调度

### 核心思想

不再按行扫描，而是将 block 分成若干 **group**，每个 group 包含相邻的几行。
group 内部按**列优先**的 Z 字形顺序调度。

```
Swizzled 调度 (GROUP_SIZE_M = 4):

      pid_n:  0    1    2    3    4    5    6    7
pid_m:   ┌────┬────┬────┬────┬────┬────┬────┬────┐
  0      │  0 │  4 │  8 │ 12 │ 16 │ 20 │ 24 │ 28 │  Group 0
         ├────┼────┼────┼────┼────┼────┼────┼────┤  (pid_m 0-3)
  1      │  1 │  5 │  9 │ 13 │ 17 │ 21 │ 25 │ 29 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  2      │  2 │  6 │ 10 │ 14 │ 18 │ 22 │ 26 │ 30 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  3      │  3 │  7 │ 11 │ 15 │ 19 │ 23 │ 27 │ 31 │
  -------├────┼────┼────┼────┼────┼────┼────┼────┤--------
  4      │ 32 │ 36 │ 40 │ 44 │ 48 │ 52 │ 56 │ 60 │  Group 1
         ├────┼────┼────┼────┼────┼────┼────┼────┤  (pid_m 4-7)
  5      │ 33 │ 37 │ 41 │ 45 │ 49 │ 53 │ 57 │ 61 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  6      │ 34 │ 38 │ 42 │ 46 │ 50 │ 54 │ 58 │ 62 │
         ├────┼────┼────┼────┼────┼────┼────┼────┤
  7      │ 35 │ 39 │ 43 │ 47 │ 51 │ 55 │ 59 │ 63 │
         └────┴────┴────┴────┴────┴────┴────┴────┘

数字 = 新的执行顺序
```

### 为什么 Swizzle 更好？

```
同时运行 pid 0-7 (GROUP_SIZE_M=4):

         pid_n:  0    1
pid_m=0:      │  0 │  4 │    这 8 个 program 需要:
pid_m=1:      │  1 │  5 │    A: A 的 第0,1,2,3行块 (4行)
pid_m=2:      │  2 │  6 │    B: B 的 第0,1列块 (只有 2列!)
pid_m=3:      │  3 │  7 │

对比朴素调度 (pid 0-7 都在 pid_m=0):
  朴素: A 的 1行块, B 的 8列块  → B 需要大量 L2 cache
  Swizzle: A 的 4行块, B 的 2列块 → A 和 B 都只需少量 L2 cache!

L2 Cache 占用对比:
  朴素:   A: 1*BM*BK*2B  +  B: 8*BK*BN*2B = 很大
  Swizzle: A: 4*BM*BK*2B +  B: 2*BK*BN*2B = 更平衡

关键: Swizzle 让同时运行的 program 访问的 B 数据更集中，
     同时 A 数据在 group 内的多行之间被复用。
```

## 13.3 Swizzle 的数学公式

### pid 重映射算法

给定一个线性 pid (0, 1, 2, ..., total_programs-1)，
我们需要重新计算 `pid_m` 和 `pid_n`：

```python
# 1D grid: pid 从 0 到 grid_m * grid_n - 1
pid = tl.program_id(0)  # 使用 1D grid

# grid 维度
grid_m = cdiv(M, BLOCK_M)  # M 方向的 block 数
grid_n = cdiv(N, BLOCK_N)  # N 方向的 block 数

# 计算 group
group_id = pid // (GROUP_SIZE_M * grid_n)      # 当前 pid 属于哪个 group
first_pid_m = group_id * GROUP_SIZE_M           # 该 group 的第一个 pid_m
group_size_m = min(grid_m - first_pid_m, GROUP_SIZE_M)  # 该 group 实际的 M 行数

# group 内的局部索引
pid_in_group = pid - group_id * (GROUP_SIZE_M * grid_n)  # 在 group 内的序号
pid_m = first_pid_m + (pid_in_group % group_size_m)      # 列优先: M 维度快变
pid_n = pid_in_group // group_size_m                      # N 维度慢变
```

### 步骤图解

```
grid_m=8, grid_n=8, GROUP_SIZE_M=4:

Group 0: pid_m = 0,1,2,3
  含 GROUP_SIZE_M * grid_n = 4 * 8 = 32 个 program (pid 0-31)
  
  pid=0: pid_in_group=0, pid_m=0+0%4=0, pid_n=0//4=0  → (0,0)
  pid=1: pid_in_group=1, pid_m=0+1%4=1, pid_n=1//4=0  → (1,0)
  pid=2: pid_in_group=2, pid_m=0+2%4=2, pid_n=2//4=0  → (2,0)
  pid=3: pid_in_group=3, pid_m=0+3%4=3, pid_n=3//4=0  → (3,0)
  pid=4: pid_in_group=4, pid_m=0+4%4=0, pid_n=4//4=1  → (0,1)  ← 列推进
  pid=5: pid_in_group=5, pid_m=0+5%4=1, pid_n=5//4=1  → (1,1)
  ...

Group 1: pid_m = 4,5,6,7  (pid 32-63)
  pid=32: group_id=1, first_pid_m=4
          pid_in_group=0, pid_m=4+0%4=4, pid_n=0  → (4,0)
  ...
```

## 13.4 实现：Swizzled GEMM

我们同时实现 swizzled 和 non-swizzled 版本，以便对比。

In [2]:
@triton.jit
def matmul_no_swizzle_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    无 Swizzle 的 GEMM kernel (朴素 block 调度)。
    使用 2D grid, 自然的行优先顺序。
    """
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    # Block Pointer 方式
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N), block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )
    
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        acc = tl.dot(a_tile, b_tile, acc=acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr, shape=(M, N), strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    tl.store(c_block_ptr, acc.to(tl.float16), boundary_check=(0, 1))

In [3]:
@triton.jit
def matmul_swizzle_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    """
    Swizzled GEMM kernel — 通过 pid 重映射优化 L2 cache 命中率。
    
    使用 1D grid, 手动计算 pid_m 和 pid_n。
    对应 CUDA 中高级的 block 调度优化 (CTA swizzle)。
    """
    # ========== Swizzled pid 重映射 ==========
    pid = tl.program_id(0)  # 1D grid
    
    grid_m = tl.cdiv(M, BLOCK_M)
    grid_n = tl.cdiv(N, BLOCK_N)
    
    # 计算 group
    group_id = pid // (GROUP_SIZE_M * grid_n)
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(grid_m - first_pid_m, GROUP_SIZE_M)
    
    # group 内的列优先映射
    pid_m = first_pid_m + ((pid % (GROUP_SIZE_M * grid_n)) % group_size_m)
    pid_n = (pid % (GROUP_SIZE_M * grid_n)) // group_size_m
    
    # ========== 后续与标准 GEMM 相同 ==========
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N), block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )
    
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        acc = tl.dot(a_tile, b_tile, acc=acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr, shape=(M, N), strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    tl.store(c_block_ptr, acc.to(tl.float16), boundary_check=(0, 1))

In [4]:
# ========== Host 端包装函数 ==========
def matmul_no_swizzle(a, b, BLOCK_M=128, BLOCK_N=128, BLOCK_K=32):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    matmul_no_swizzle_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return c

def matmul_swizzle(a, b, BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, GROUP_SIZE_M=8):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N),)  # 1D grid!
    matmul_swizzle_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        GROUP_SIZE_M=GROUP_SIZE_M,
    )
    return c

In [5]:
# ========== 正确性验证 ==========
torch.manual_seed(42)
M, N, K = 2048, 2048, 1024
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

c_no_swizzle = matmul_no_swizzle(a, b)
c_swizzle = matmul_swizzle(a, b)
c_ref = torch.matmul(a, b)

print("正确性验证:")
print(f"  无Swizzle vs cuBLAS: max_err={( c_no_swizzle - c_ref).abs().max().item():.4f}")
print(f"  有Swizzle vs cuBLAS: max_err={(c_swizzle - c_ref).abs().max().item():.4f}")
print(f"  两版本一致: {torch.allclose(c_no_swizzle, c_swizzle, atol=0.01)}")

正确性验证:
  无Swizzle vs cuBLAS: max_err=0.0000
  有Swizzle vs cuBLAS: max_err=0.0000
  两版本一致: True


In [6]:
# ========== Swizzle vs No-Swizzle 性能对比 ==========
def benchmark_gemm(fn, a, b, num_warmup=10, num_rep=50, **kwargs):
    """通用 benchmark 函数。"""
    for _ in range(num_warmup):
        fn(a, b, **kwargs)
    torch.cuda.synchronize()
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(num_rep):
        fn(a, b, **kwargs)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / num_rep

print("Swizzle vs No-Swizzle 性能对比")
print(f"{'M':>6} {'N':>6} {'K':>6} | {'无Swizzle(ms)':>14} {'有Swizzle(ms)':>14} | {'加速比':>8}")
print("-" * 70)

for M, N, K in [
    (1024, 1024, 1024),
    (2048, 2048, 1024),
    (2048, 2048, 2048),
    (4096, 4096, 1024),
    (4096, 4096, 4096),
]:
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    ms_no = benchmark_gemm(matmul_no_swizzle, a, b)
    ms_sw = benchmark_gemm(matmul_swizzle, a, b, GROUP_SIZE_M=8)
    print(f"{M:>6} {N:>6} {K:>6} | {ms_no:>14.3f} {ms_sw:>14.3f} | {ms_no/ms_sw:>7.2f}x")

Swizzle vs No-Swizzle 性能对比
     M      N      K |   无Swizzle(ms)   有Swizzle(ms) |      加速比
----------------------------------------------------------------------
  1024   1024   1024 |          0.064          0.040 |    1.59x
  2048   2048   1024 |          0.035          0.037 |    0.94x
  2048   2048   2048 |          0.062          0.062 |    1.00x
  4096   4096   1024 |          0.102          0.118 |    0.86x
  4096   4096   4096 |          0.379          0.391 |    0.97x


## 13.5 GROUP_SIZE_M 的选择

GROUP_SIZE_M 控制了 L2 cache 的平衡：

```
GROUP_SIZE_M = 1: 等价于朴素调度 (按行扫描)
  → A 缓存好, B 缓存差

GROUP_SIZE_M = grid_m: 等价于列优先调度
  → B 缓存好, A 缓存差

GROUP_SIZE_M = sqrt(grid_m): 通常接近最优
  → A 和 B 的缓存使用量平衡

L2 Cache 使用量分析:
  同时运行 GROUP_SIZE_M * min(SM数/GROUP_SIZE_M, grid_n) 个 program
  A 需要: GROUP_SIZE_M * BLOCK_M * BLOCK_K * sizeof(half)
  B 需要: ceil(SM数/GROUP_SIZE_M) * BLOCK_K * BLOCK_N * sizeof(half)
  
  总量 = GROUP_SIZE_M * BM * BK + ceil(SM/GROUP_SIZE_M) * BK * BN
  对 GROUP_SIZE_M 求最优 → 约为 sqrt(SM * BM / BN)
```

实际中，**GROUP_SIZE_M = 8** 在大多数情况下是一个不错的默认值。

In [7]:
# ========== 不同 GROUP_SIZE_M 的性能 ==========
M, N, K = 4096, 4096, 2048
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

print(f"矩阵规模: M={M}, N={N}, K={K}")
print(f"grid: ({triton.cdiv(M, 128)}, {triton.cdiv(N, 128)})")
print(f"\n{'GROUP_SIZE_M':>15} | {'时间(ms)':>10} | {'TFLOPS':>8}")
print("-" * 45)

flops = 2.0 * M * N * K
for gsm in [1, 2, 4, 8, 16, 32]:
    ms = benchmark_gemm(matmul_swizzle, a, b, GROUP_SIZE_M=gsm)
    tflops = flops / (ms * 1e-3) / 1e12
    print(f"{gsm:>15} | {ms:>10.3f} | {tflops:>8.2f}")

矩阵规模: M=4096, N=4096, K=2048
grid: (32, 32)

   GROUP_SIZE_M |     时间(ms) |   TFLOPS
---------------------------------------------
              1 |      0.193 |   356.77
              2 |      0.190 |   360.78
              4 |      0.210 |   328.00
              8 |      0.203 |   339.25
             16 |      0.203 |   337.89
             32 |      0.203 |   337.98


## 13.6 make_block_ptr 的 order 参数

`order` 参数不仅影响全局内存访问，还影响 Shared Memory 的布局：

```python
# 行主序: 最内层维度 (列) 连续
a_block_ptr = tl.make_block_ptr(
    ..., order=(1, 0),  # 维度 1 (列) 最内层
)

# 列主序: 最内层维度 (行) 连续
a_block_ptr = tl.make_block_ptr(
    ..., order=(0, 1),  # 维度 0 (行) 最内层
)
```

### order 对 Shared Memory 布局的影响

```
order=(1, 0) → 行主序 Shared Memory 布局:
  smem:  [row0_col0, row0_col1, row0_col2, ..., row1_col0, row1_col1, ...]
  适合: B 矩阵 (行主序, 需要按行读取)

order=(0, 1) → 列主序 Shared Memory 布局:
  smem:  [row0_col0, row1_col0, row2_col0, ..., row0_col1, row1_col1, ...]
  适合: A 矩阵的转置存储 (类似 CUDA simt_smemT 的做法!)
```

这与 CUDA `simt_smemT.cu` 的优化思路完全对应：
- CUDA 中手动将 A 矩阵转置后存入 Shared Memory
- Triton 中通过 `order` 参数让编译器自动选择最优布局

### 对应 CUDA simt_smemT 的关键代码

```cpp
// simt_smemT.cu: A 矩阵转置存储到 smem
// 读取: row-major 从 global memory
float4 buffer = gmem_blockA_f4_ptr[k/8 + ay * ldm_A_f4size + ax];

// 写入: 转置后存储到 smem (交换了 ax 和 ay)
smem_blockA_hf_ptr[(ay * 8 + e) * BlockTileM + ax]  // 注意: ax 和 ay 互换!

// 效果: 后续按列读取 smem 时, 数据是连续的 → 无 bank conflict
```

在 Triton 中，只需要改一个参数就能达到相同效果。

## 13.7 完整的 Swizzled GEMM (带 cuBLAS 对比)

让我们做一个完整的性能对比，包括 cuBLAS。

In [8]:
# ========== 完整性能对比 ==========
def benchmark_cublas(a, b, num_warmup=10, num_rep=50):
    for _ in range(num_warmup):
        torch.matmul(a, b)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(num_rep):
        torch.matmul(a, b)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / num_rep

print("完整性能对比: No-Swizzle vs Swizzle vs cuBLAS")
print(f"{'Size':>20} | {'NoSwizzle':>10} {'Swizzle':>10} {'cuBLAS':>10} | {'Sw/NoSw':>8} {'Sw/cuBLAS':>10}")
print("-" * 80)

for M, N, K in [
    (1024, 1024, 1024),
    (2048, 2048, 1024),
    (2048, 2048, 2048),
    (4096, 4096, 2048),
    (4096, 4096, 4096),
]:
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    ms_no = benchmark_gemm(matmul_no_swizzle, a, b)
    ms_sw = benchmark_gemm(matmul_swizzle, a, b, GROUP_SIZE_M=8)
    ms_cu = benchmark_cublas(a, b)
    
    size_str = f"{M}x{N}x{K}"
    print(f"{size_str:>20} | {ms_no:>9.3f}ms {ms_sw:>9.3f}ms {ms_cu:>9.3f}ms | {ms_no/ms_sw:>7.2f}x {ms_cu/ms_sw:>9.2f}x")

完整性能对比: No-Swizzle vs Swizzle vs cuBLAS
                Size |  NoSwizzle    Swizzle     cuBLAS |  Sw/NoSw  Sw/cuBLAS
--------------------------------------------------------------------------------
      1024x1024x1024 |     0.028ms     0.043ms     0.011ms |    0.66x      0.26x
      2048x2048x1024 |     0.034ms     0.034ms     0.033ms |    1.01x      0.99x
      2048x2048x2048 |     0.062ms     0.067ms     0.060ms |    0.93x      0.90x
      4096x4096x2048 |     0.199ms     0.197ms     0.225ms |    1.01x      1.14x
      4096x4096x4096 |     0.389ms     0.401ms     0.458ms |    0.97x      1.14x


## 13.8 总结

### 本章要点

1. **L2 Cache 抖动问题**：朴素的行优先 block 调度导致同时运行的 program 需要 B 矩阵的大量不同列，超过 L2 cache 容量

2. **Swizzle 解决方案**：将 block 分组 (GROUP_SIZE_M)，组内按列优先调度，使得同时运行的 program 访问的数据更集中

3. **实现细节**：
   - 使用 1D grid 替代 2D grid
   - 通过 group_id, first_pid_m, group_size_m 计算 swizzled 的 pid_m 和 pid_n
   - GROUP_SIZE_M = 8 是常用的默认值

4. **order 参数**：
   - `(1, 0)` = 行主序，列维度连续
   - `(0, 1)` = 列主序，行维度连续
   - 影响 Shared Memory 的数据布局，对应 CUDA 中的转置存储优化

5. **与 CUDA 的对应**：
   | Triton Swizzle | CUDA 对应 |
   |----------------|----------|
   | GROUP_SIZE_M | CTA swizzle / grid 调度优化 |
   | order=(0,1) | simt_smemT 的 smem 转置存储 |
   | 自动 bank conflict 避免 | 手动 padding/swizzle |

### 练习

1. **可视化 Swizzle 模式**：对于 grid_m=16, grid_n=16, 分别画出 GROUP_SIZE_M=1,4,8,16 的调度顺序
2. **L2 cache 分析**：用 `triton.testing.do_bench` 和 NCU profiler 观察 L2 cache 命中率的变化
3. **非方阵矩阵**：测试 M >> N (如 8192x256) 和 M << N (如 256x8192) 时 Swizzle 的效果
4. **思考题**：当 grid_m 不是 GROUP_SIZE_M 的整数倍时，最后一个 group 的行为是什么？

### 下一章预告

第14章将介绍 **软件流水线 (Software Pipelining)**，通过重叠数据加载和计算来隐藏内存延迟，
这直接对应 CUDA 项目中 `simt_pipline.cu` 的双缓冲/三缓冲优化。